# Debate Consensus

> Notebook generated from [`examples/subgraphs/08_debate_consensus.py`](../../examples/subgraphs/08_debate_consensus.py).

| Item | Value |
|------|-------|
| **Dataset** | AI Policy & Tech Ethics (embedded) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** proponent → opponent → moderator → consensus. Arguments per round + Jaccard agreement score.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Debate Consensus Subgraph — Position synthesis via structured debate
===============================================================================
Subgraph: prismal.agents.subgraphs.debate_consensus

Dataset: AI Policy & Tech Ethics (technology policy debates)
  • Topics: AI regulation, open-source language models, privacy vs
    personalization, algorithmic hiring, copyright in generative AI.
  • Reference: Stanford Encyclopedia of Philosophy (AI Ethics), EU AI Act,
    ACM FAccT conference debates, UNESCO Recommendation on AI (2021).
  • Why: Tech policy debates are ideal because they have
    legitimate arguments on both sides (proponent/opponent), require nuanced
    synthesis (moderator), and produce positions verifiable via Jaccard
    agreement (consensus). AI ethics topics are well known to LLMs
    and let us evaluate the quality of reasoning.

Debate Consensus subgraph description:
  proponent → opponent → moderator → consensus

  Nodes:
  1. proponent — builds the strongest argument IN FAVOR of the thesis
  2. opponent  — builds the strongest argument AGAINST the thesis
  3. moderator — evaluates both positions, identifies points of agreement and tension
  4. consensus — synthesizes a balanced position with Jaccard agreement score

  Reuses primitives from prismal.agents.patterns.debate:
    DebatePosition, pairwise_jaccard()

Usage:
    uv run python examples/subgraphs/08_debate_consensus.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

# Import primitives of the debate pattern
try:
    from prismal.agents.patterns.debate import DebatePosition, pairwise_jaccard

    DEBATE_PRIMITIVES_AVAILABLE = True
except ImportError:
    DEBATE_PRIMITIVES_AVAILABLE = False

# Import the subgraph
try:
    from prismal.agents.subgraphs.debate_consensus.builder import (
        build_debate_consensus_subgraph,
        register_debate_consensus,
    )

    DEBATE_CONSENSUS_AVAILABLE = True
except ImportError:
    DEBATE_CONSENSUS_AVAILABLE = False

## Dataset: technology policy debate topics

In [ ]:
DEBATE_TOPICS = [
    {
        "id": "DEB-001",
        "thesis": "Governments should strictly regulate AI development through "
        "mandatory licensing and independent audits before deployment.",
        "domain": "AI Regulation",
        "context": (
            "Reference: EU AI Act (2024), GPAI governance proposals, debates in the "
            "U.S. Congress on the AI Act 2024."
        ),
        "expected_agreement": "medium",  # clear points of agreement + disagreement
    },
    {
        "id": "DEB-002",
        "thesis": "Large language models should be fully "
        "open-source, with public weights and training data.",
        "domain": "Open Source AI",
        "context": (
            "Meta Llama vs OpenAI GPT-4. Debate between Yann LeCun and Sam Altman on "
            "openness vs safety in LLMs (2023-2024)."
        ),
        "expected_agreement": "low",  # highly opposed positions
    },
    {
        "id": "DEB-003",
        "thesis": "Use of AI in hiring decisions should be "
        "banned until algorithmic bias is demonstrated to be < 5%.",
        "domain": "Algorithmic Hiring",
        "context": (
            "Amazon disabled its AI hiring system in 2018 due to gender bias. "
            "NYC Local Law 144 (2023) requires bias audits for HR AI."
        ),
        "expected_agreement": "high",  # consensus possible with conditions
    },
    {
        "id": "DEB-004",
        "thesis": "Training generative AI models on copyright-protected works "
        "should require a license and compensation to the original authors.",
        "domain": "AI Copyright",
        "context": (
            "Cases: Getty Images vs Stability AI, New York Times vs OpenAI (2023). "
            "Debate on fair use, transformative use, and creators' rights."
        ),
        "expected_agreement": "medium",
    },
]

## Pre-elaborated arguments (simulates what the LLM would generate)

In [ ]:
DEBATE_ARGUMENTS = {
    "DEB-001": {
        "proponent": [
            "High-capability AI poses systemic risks comparable to nuclear energy — it needs government oversight.",
            "Without regulation, the AI arms race between companies will sacrifice safety for speed.",
            "Citizens have the right to know which AI systems make decisions that affect them.",
            "Independent audits create incentives for safety-by-design rather than safety-as-afterthought.",
            "The EU AI Act shows that regulation and innovation are compatible.",
        ],
        "opponent": [
            "Mandatory licensing would create barriers to entry that concentrate power in a few established companies.",
            "AI innovation is too fast for static regulatory frameworks — they would be obsolete before implementation.",
            "Excessive regulation will push development to less-regulated jurisdictions, worsening the problem.",
            "Governments lack the technical expertise to meaningfully audit complex AI systems.",
            "Industry self-regulation has worked for the internet — the same model can apply to AI.",
        ],
        "agreement_points": [
            "Basic transparency about high-risk AI systems is necessary.",
            "Some level of accountability for AI-caused harm is reasonable.",
        ],
        "tension_points": [
            "Who defines what counts as 'high risk'?",
            "How to prevent regulation from favoring incumbents?",
        ],
    },
    "DEB-002": {
        "proponent": [
            "Open source democratizes access to AI, preventing power concentration in a few corporations.",
            "Full transparency enables independent security audits that improve trust.",
            "Open-source models like Llama 2/3 have accelerated research without major safety incidents.",
            "Control over training data lets organizations comply with GDPR and other privacy regulations.",
            "Competition in open-source AI is the best defense against tech monopoly.",
        ],
        "opponent": [
            "Publishing weights of models capable of bioweapon synthesis or cyberattacks is irreversible — the risk is permanent.",
            "Malicious actors (states, terrorist groups) benefit disproportionately from unrestricted access.",
            "Training data often contains PII, proprietary information, or illegal content — publishing it creates new legal issues.",
            "Full openness removes the safety alignment mechanisms that require controlled access.",
            "Meta and other 'open-source' AI keep control over commercial fine-tuning — it is not truly open.",
        ],
        "agreement_points": [
            "Transparency about architecture and training methodology is beneficial.",
            "Lower-capability models can be open-sourced with lower risk.",
        ],
        "tension_points": [
            "Where to draw the capability line for open access?",
            "Who decides what is 'safe enough' to publish?",
        ],
    },
    "DEB-003": {
        "proponent": [
            "AI hiring systems have been shown to reproduce and amplify gender, race, and age biases.",
            "A temporary ban until fairness is validated protects historically discriminated groups.",
            "NYC Local Law 144 already sets a viable precedent for regulation with a measurable threshold.",
            "Candidates have the right to know whether AI rejected them and why.",
            "The cost of bias auditing is negligible vs the harm to discriminated candidates.",
        ],
        "opponent": [
            "Human hiring processes are more biased than properly audited algorithms.",
            "A total ban prevents iterative improvements — audit, not prohibition, should be required.",
            "The 5% threshold is arbitrary and lacks a consensual scientific basis.",
            "Many startups cannot afford independent audits, which favors large corporations.",
            "AI can detect success patterns that human judgment would unfairly ignore.",
        ],
        "agreement_points": [
            "AI hiring systems must be audited for bias before deployment.",
            "Candidates must have the right to appeal automated decisions.",
            "Transparency about AI's role in the process is necessary.",
        ],
        "tension_points": [
            "Prohibition or regulation with strict requirements?",
            "Which 'bias' metric to use and who validates it?",
        ],
    },
    "DEB-004": {
        "proponent": [
            "Training on copyrighted works without compensation is intellectual theft at massive scale.",
            "Generative models produce economic value directly derived from others' creative work.",
            "A licensing system like music's (ASCAP/BMI) is technically feasible for AI.",
            "Without compensation, original content creation becomes economically unviable.",
            "The NYT vs OpenAI case suggests courts will back creators' rights.",
        ],
        "opponent": [
            "Human learning is also based on consuming copyrighted works — AI models do the same.",
            "'Fair use' for transformative use is well established in U.S. jurisprudence.",
            "It is technically infeasible to trace which specific works contributed to which AI outputs.",
            "A massive licensing system would paralyze academic NLP research.",
            "Models do not 'copy' — they extract statistical patterns; they do not reproduce works.",
        ],
        "agreement_points": [
            "Reproducing protected works directly (memorization) violates copyright.",
            "Some opt-out mechanism for creators is reasonable.",
        ],
        "tension_points": [
            "Where does legitimate learning end and infringement begin?",
            "How to implement compensation at the scale of billions of parameters?",
        ],
    },
}


def simple_jaccard(text_a: str, text_b: str) -> float:
    """Calculate Jaccard similarity between two texts (proxy for agreement)."""
    words_a = set(text_a.lower().split())
    words_b = set(text_b.lower().split())
    if not words_a and not words_b:
        return 1.0
    intersection = len(words_a & words_b)
    union = len(words_a | words_b)
    return intersection / union if union > 0 else 0.0


def simulate_moderator(args: dict) -> dict:
    """Simulate the moderator node: evaluate both positions."""
    proponent_args = args["proponent"]
    opponent_args = args["opponent"]

    # Compute Jaccard between positions (the lower, the more divergent)
    pro_text = " ".join(proponent_args)
    con_text = " ".join(opponent_args)

    # Use pairwise_jaccard if available
    if DEBATE_PRIMITIVES_AVAILABLE:
        jaccard = pairwise_jaccard(proponent_args, opponent_args)
    else:
        jaccard = simple_jaccard(pro_text, con_text)

    return {
        "jaccard_agreement": jaccard,
        "agreement_points": args.get("agreement_points", []),
        "tension_points": args.get("tension_points", []),
        "proponent_strength": min(0.5 + len(proponent_args) * 0.08, 0.95),
        "opponent_strength": min(0.5 + len(opponent_args) * 0.08, 0.95),
    }


def simulate_consensus(topic: dict, moderation: dict) -> str:
    """Simulate the consensus node: generate a balanced position."""
    agreement_pts = moderation.get("agreement_points", [])
    tension_pts = moderation.get("tension_points", [])
    jaccard = moderation["jaccard_agreement"]

    if jaccard > 0.15:
        stance = "partially in agreement with progressive regulation"
    elif jaccard > 0.08:
        stance = "requires case-by-case contextual analysis"
    else:
        stance = "polarized, with no clear consensus — the debate remains open"

    return (
        f"After analyzing both positions, the synthesis indicates that the debate is {stance}. "
        f"There are {len(agreement_pts)} points of convergence and {len(tension_pts)} unresolved tensions. "
        f"The Jaccard agreement score ({jaccard:.3f}) reflects "
        f"{'considerable conceptual overlap' if jaccard > 0.1 else 'fundamentally divergent positions'}."
    )


async def run_debate(topic: dict) -> dict:
    """Run the debate_consensus pipeline for a topic."""
    print(f"\n[{topic['id']}] {topic['domain'].upper()}")
    print(f"  Thesis : {topic['thesis'][:80]}...")
    print(f"  Context: {topic['context'][:80]}...")

    args = DEBATE_ARGUMENTS[topic["id"]]

    if not DEBATE_CONSENSUS_AVAILABLE:
        print("  [Demo mode — simulated subgraph]")

        # Node 1: proponent
        print("\n  ── Node 1: proponent ──")
        print(f"    FOR arguments ({len(args['proponent'])}):")
        for arg in args["proponent"][:3]:
            print(f"      + {arg[:70]}")

        # Node 2: opponent
        print("\n  ── Node 2: opponent ──")
        print(f"    AGAINST arguments ({len(args['opponent'])}):")
        for arg in args["opponent"][:3]:
            print(f"      - {arg[:70]}")

        # Node 3: moderator
        print("\n  ── Node 3: moderator ──")
        moderation = simulate_moderator(args)
        jaccard = moderation["jaccard_agreement"]
        print(f"    Jaccard agreement score: {jaccard:.4f}")
        agreement_level = "HIGH" if jaccard > 0.15 else "MEDIUM" if jaccard > 0.08 else "LOW"
        bar = "█" * int(jaccard * 100)
        print(f"    Agreement level        : {agreement_level}  {bar}")
        print(f"    Points in common ({len(moderation['agreement_points'])}):")
        for pt in moderation["agreement_points"]:
            print(f"      ≈ {pt[:70]}")
        print(f"    Unresolved tensions ({len(moderation['tension_points'])}):")
        for pt in moderation["tension_points"]:
            print(f"      ⚡ {pt[:70]}")

        # Node 4: consensus
        print("\n  ── Node 4: consensus ──")
        consensus_text = simulate_consensus(topic, moderation)
        print(f"    {consensus_text}")

        # Verify the agreement expectation
        expected = topic["expected_agreement"]
        actual_level = agreement_level.lower()
        match = "✓" if actual_level == expected else "~"
        print(f"\n  Expected agreement: {expected} | Obtained: {actual_level} {match}")

        return {
            "id": topic["id"],
            "domain": topic["domain"],
            "jaccard": jaccard,
            "agreement_level": agreement_level,
            "consensus": consensus_text,
            "agreement_points": len(moderation["agreement_points"]),
            "tension_points": len(moderation["tension_points"]),
        }

    # Real mode with LangGraph subgraph
    from langchain_core.messages import HumanMessage

    from prismal.agents.state import initial_state

    await register_debate_consensus()
    subgraph = build_debate_consensus_subgraph()

    state = initial_state()
    state["messages"] = [
        HumanMessage(
            content=(
                f"Debate on the following thesis:\n{topic['thesis']}\n\n"
                f"Context: {topic['context']}"
            )
        )
    ]
    state["metadata"] = {
        "debate_consensus": {
            "thesis": topic["thesis"],
            "domain": topic["domain"],
        }
    }

    config = {"configurable": {"thread_id": f"debate_{topic['id']}_001"}}
    final_state = await subgraph.graph.ainvoke(state, config=config)

    debate_meta = final_state.get("metadata", {}).get("debate_consensus", {})
    messages = final_state.get("messages", [])
    return {
        "id": topic["id"],
        "domain": topic["domain"],
        "jaccard": debate_meta.get("jaccard_score", 0.0),
        "consensus": str(messages[-1].content) if messages else "",
        "agreement_points": len(debate_meta.get("agreement_points", [])),
        "tension_points": len(debate_meta.get("tension_points", [])),
    }


async def main() -> None:
    print("=" * 70)
    print("  Debate Consensus Subgraph — Dataset: AI Policy & Tech Ethics")
    print("=" * 70)

    print("\n[Debate Consensus subgraph architecture]")
    print("  proponent  → builds the STRONGEST argument in favor")
    print("       ↓")
    print("  opponent   → builds the STRONGEST argument against")
    print("       ↓")
    print("  moderator  → evaluates both positions; computes Jaccard agreement")
    print("       ↓")
    print("  consensus  → synthesizes a balanced position with agreement score")
    print()
    print("  Based on: prismal.agents.patterns.debate")
    print("  Score: pairwise_jaccard(pro_args, con_args) ∈ [0, 1]")
    print("    → 0.0 = fully opposed positions")
    print("    → 1.0 = identical positions")

    print(f"\n[Debating {len(DEBATE_TOPICS)} technology policy topics]")
    results = []
    for topic in DEBATE_TOPICS:
        result = await run_debate(topic)
        results.append(result)
        print("─" * 70)

    # ── Global statistics ─────────────────────────────────────────────────────
    print("\n[Statistical summary — all debates]")
    print(
        f"\n  {'ID':<10} {'Domain':<25} {'Jaccard':>8} {'Agreement':>10} {'Common':>7} {'Tension':>8}"
    )
    print("  " + "─" * 72)
    for r in results:
        print(
            f"  {r['id']:<10} {r['domain']:<25} {r['jaccard']:>8.4f} "
            f"{r['agreement_level']:>10} {r['agreement_points']:>7} {r['tension_points']:>8}"
        )

    avg_jaccard = sum(r["jaccard"] for r in results) / len(results)
    print(f"\n  Average Jaccard      : {avg_jaccard:.4f}")
    print(f"  Most polarized topic : {min(results, key=lambda r: r['jaccard'])['domain']}")
    print(f"  Most consensual topic: {max(results, key=lambda r: r['jaccard'])['domain']}")

    # ── Comparison: Debate Pattern vs Debate Consensus Subgraph ─────────────
    print("\n[Debate Pattern vs Debate Consensus Subgraph]")
    comparison = [
        ("debate.py (pattern)", "N agents", "M rounds", "Jaccard + synthesis", "Flexible"),
        (
            "debate_consensus (subgraph)",
            "2 fixed roles",
            "1 round",
            "Jaccard + consensus",
            "Structured",
        ),
    ]
    header = f"  {'Component':<28} {'Agents':<14} {'Rounds':<10} {'Score':<22} {'Use'}"
    print(header)
    print("  " + "─" * 80)
    for name, agents, rounds, score, use in comparison:
        print(f"  {name:<28} {agents:<14} {rounds:<10} {score:<22} {use}")

    print("\n[When to use Debate Consensus Subgraph]")
    use_cases = [
        ("✓", "Complex binary decisions (adopt technology X vs Y)"),
        ("✓", "Risk analysis with multiple perspectives"),
        ("✓", "Generating balanced content on controversial topics"),
        ("✓", "Validating architecture or design proposals"),
        ("✓", "Due diligence: pros/cons of an acquisition or partnership"),
        ("✗", "Topics with an objectively correct answer (math, facts)"),
        ("✗", "When > 2 positions are required — use debate.py with N agents"),
    ]
    for mark, case in use_cases:
        print(f"  {mark} {case}")

    print("\n[Integration with the full Debate pattern]")
    print("  # For more complex debates with N agents and M rounds:")
    print("  from prismal.agents.patterns.debate import debate_round")
    print("  result = await debate_round(")
    print("      query=thesis,")
    print("      state=state,")
    print("      n_agents=5,    # proponent, opponent, + 3 specialists")
    print("      n_rounds=3,    # rebuttal and counter-rebuttal")
    print("      roles=['economist', 'ethicist', 'engineer', 'regulator', 'citizen'],")
    print("      synthesis_strategy='moderator',")
    print("  )")
    print("  # result.agreement_score: average Jaccard across all positions")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()